# 01 · Teacher & Data — the license-audited FMA pipeline and what teacher labels look like

This notebook walks the Direction-10 data engine end to end: the **license audit** (allowlist,
DENY-by-default, the load-bearing ND-exclusion), the **deterministic screen sample**, the
committed **track-ID manifest** (never audio), the **teacher-labeling** cells (RUN LATER,
2–4 GPU-h), the **2-stem consistency** + recorded 4-stem residual, the **vocal-activity screen**
(Direction 08's machinery, with the pre-registered 10 % fallback), a **pseudo-stem gallery**
(RUN LATER), and the **leakage-guard story** (the pseudo pool structurally refuses MUSDB shard
paths). Everything CPU-runnable now uses synthetic fixtures; nothing downloads FMA or demucs.

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §3 (the pipeline), §3.3 (leakage guards),
  §6 (run book + runtimes); [`../THEORY.md`](../THEORY.md) §5 (the license formalism — the crux),
  §2/§3.2 (2-stem consistency + the teacher residual).
- **MUSDB data prep is *not* repeated here** — reused from Direction 01
  ([`../../01-loss-function-study/notebooks/01_data_and_eda.ipynb`](../../01-loss-function-study/notebooks/01_data_and_eda.ipynb)).
  This direction adds only the FMA metadata pass (`scripts/prepare_fma.py`) and the teacher
  labeling (`scripts/teacher_label.py`).
- **Un-run by design:** every heavy cell is a ⚠️ RUN-LATER banner; the CPU cells render tested
  logic on synthetic fixtures. Nothing here touches FMA audio, demucs weights, or the network.

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU. Installs the pinned env and mounts Drive.
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt          # + `pip install demucs` for the teacher (RUN LATER)
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"]  = "/content/drive/MyDrive/musdb_shards"
# os.environ["PSEUDO_ROOT"] = "/content/drive/MyDrive/fma_pseudo"   # SEPARATE root (§3.3 guard)
import sys
from pathlib import Path
# scripts/ holds the committed, unit-tested pipeline (prepare_fma, teacher_label); add it to the
# path exactly as tests/ do, so the notebook IMPORTS that tested logic instead of redefining it.
sys.path.insert(0, str(Path.cwd() / "scripts"))
print("Bootstrap cell — run on Colab only (see comments). No-op here.")

## 1 · License audit — allowlist, DENY-by-default, the ND-exclusion (THEORY §5)

FMA's metadata is CC BY 4.0 but its **audio is per-track, artist-chosen CC** — so a license-safe
subset is filtered on the `tracks.csv` `license` field. The decision is an **allowlist**
$\mathcal A=\{$CC0/PD, CC-BY, CC-BY-SA, CC-BY-NC, CC-BY-NC-SA$\}$ with **NoDerivatives (ND) as a
hard veto** (separated stems are derivative works — the load-bearing legal call) and
**DENY-by-default** for anything unclear. The cell below renders the full allow/deny table on a
fixture metadata frame — the same table `01_teacher_and_data` would render on the real
`tracks.csv` (CPU-runnable now).

In [ ]:
# CPU-runnable now: the license allow/deny table on FIXTURE metadata (no FMA download).
import pandas as pd
from prepare_fma import license_audit, ALLOWED_LICENSES   # committed, unit-tested (tests/test_prepare_fma.py)

fixture = pd.DataFrame([
    {"track_id": "10", "license": "Attribution-NonCommercial-ShareAlike 3.0"},  # allow -> CC-BY-NC-SA
    {"track_id": "11", "license": "CC0 1.0 Universal"},                          # allow -> CC0
    {"track_id": "12", "license": "Attribution 3.0 United States"},              # allow -> CC-BY
    {"track_id": "13", "license": "http://creativecommons.org/licenses/by-nc/3.0/"},  # allow -> CC-BY-NC
    {"track_id": "14", "license": "Attribution-NoDerivatives 4.0 International"}, # DENY (ND, the veto)
    {"track_id": "15", "license": "Attribution-NonCommercial-NoDerivatives 4.0"},# DENY (NC-ND)
    {"track_id": "16", "license": "All Rights Reserved"},                         # DENY (non-CC)
    {"track_id": "17", "license": ""},                                            # DENY (empty)
    {"track_id": "18", "license": "some proprietary eula"},                       # DENY (unclear)
])
audit = license_audit(fixture)
print("pinned allowlist:", sorted(ALLOWED_LICENSES))
print(f"allowed {int(audit['allowed'].sum())} / {len(audit)}  "
      f"(NC included for non-commercial research; every -ND denied — THEORY §5)")
audit[["track_id", "license", "license_canonical", "allowed"]]

## 2 · Deterministic screen sample (MASTER_PLAN §3.1)

From the allowlisted pool we draw `N_screen = 1200` clips **deterministically** in `(ids, n, seed)`
— input-order independent, reproducible, seed-sensitive. This is the frozen screen the teacher
later labels; freezing it (gate G0b) is what makes the manifest a stable audit artifact.

In [ ]:
# CPU-runnable now: the screen sample is deterministic and reproducible (no FMA needed).
from prepare_fma import screen_sample

pool = [str(i) for i in range(2000)]
a = screen_sample(pool, 1200, seed=0)
b = screen_sample(pool, 1200, seed=0)
c = screen_sample(pool, 1200, seed=1)
print(f"same seed reproducible: {a == b}")
print(f"different seed differs: {a != c}")
print(f"input-order independent: {screen_sample(pool, 50, 0) == screen_sample(list(reversed(pool)), 50, 0)}")
print(f"drew {len(a)} unique screened ids (subset of the allowlisted pool)")

## 3 · The committed manifest — IDs + licenses + screen flags, **never audio**

`prepare()` license-filters, screen-samples, and assembles the manifest frame; `write_manifest`
adds a `#` provenance header (the allowlist + screen seed + counts). This is the **only** FMA
artifact committed — a small text table for public audit. No audio, ever.

In [ ]:
# CPU-runnable now: the manifest frame + audit counts on the fixture (mirrors the real run).
from prepare_fma import prepare

manifest, counts = prepare(fixture, screen=3, seed=0)
print("counts:", counts, "  (NO AUDIO is ever written — only this table)")
manifest

## 4 · Teacher labeling — the htdemucs invocation (⚠️ RUN LATER, 2–4 GPU-h once)

The teacher (`htdemucs`, the engine StemCraft ships) labels the screened clips **once**. The
demucs invocation is constructed in `scripts/teacher_label.py` behind a **lazy, RUN-LATER-guarded
import** (importing the module pulls in no heavy deps). The provenance file (version / model /
settings / mean consistency residual) is committed.

In [ ]:
# ⚠️ RUN THIS LATER — the FMA metadata pass (CPU ~1 h) + teacher labeling (GPU 2–4 h once).
# Prereqs: the FMA metadata + audio on Drive; `pip install demucs`. Nothing runs here.
#
#   # 0. license filter + screen + manifest (CPU+network, ~1 h; commits the manifest):
#   !python scripts/prepare_fma.py --metadata-dir $FMA_META --audio-dir $FMA_AUDIO \
#          --screen 1200 --seed 0 --out $PSEUDO_ROOT
#   # 1. teacher labeling + activity screen (GPU 2–4 h, once; writes shards + provenance):
#   !python scripts/teacher_label.py --manifest $PSEUDO_ROOT/manifest.csv \
#          --keep 800 --activity-threshold 0.20 --out $PSEUDO_ROOT
print("RUN-LATER banner — teacher labeling is a one-time GPU pass (MASTER_PLAN §6). No-op here.")

## 5 · 2-stem consistency + the recorded 4-stem residual (THEORY §2, §3.2)

The pinned choice: `vocals = teacher_vocals`; `accompaniment := mixture − teacher_vocals`
(**exact** additivity, mirroring the student's task). The teacher's raw 4-stem sum-residual is
**measured and recorded** (`consistency_residual_db`) — a provenance number, never a target. Both
are pure functions, exercised here on synthetic arrays.

In [ ]:
# CPU-runnable now: â = x − v̂ is exact; the 4-stem residual is recorded, not used.
import numpy as np
from teacher_label import two_stem_consistency, consistency_residual_db

rng = np.random.default_rng(0)
mixture = rng.standard_normal(6 * 44100).astype(np.float32)
teacher_vocals = 0.4 * rng.standard_normal(6 * 44100).astype(np.float32)
two = two_stem_consistency(mixture, teacher_vocals)
additivity_err = float(np.max(np.abs((two["vocals"] + two["accompaniment"]) - mixture)))
print(f"max |(v̂ + â) − x| = {additivity_err:.2e}  (float32 round-off — exact by construction)")

# a self-consistent teacher (stems sum to x) -> very negative residual; an inconsistent one -> ~0 dB.
consistent = {"drums": rng.standard_normal(1024), "bass": rng.standard_normal(1024),
              "other": rng.standard_normal(1024), "vocals": rng.standard_normal(1024)}
x = sum(consistent.values())
print(f"4-stem consistency residual (self-consistent teacher): {consistency_residual_db(consistent, x):.1f} dB")

## 6 · Vocal-activity screen — Direction 08's machinery + the 10 % fallback (§3.2)

Using the teacher vocals' windowed-RMS profile (`singnet.data.profiles.windowed_vocal_rms`, the
D08 tooling), each clip gets an **activity ratio**; we keep clips with ratio ≥ 0.20 and take the
first `N_train = 800`. **Pre-registered fallback:** if fewer than 800 survive, lower the threshold
to 0.10 **once**; if still short, use all survivors and record the count.

In [ ]:
# CPU-runnable now: the activity ratio (D08 profile) + the screen with its one-shot fallback.
from teacher_label import vocal_activity_ratio, screen_by_activity

sr = 44100
t = np.arange(12 * sr) / sr
loud = (0.3 * np.sin(2 * np.pi * 220 * t)).astype(np.float32)          # sustained -> ratio 1.0
half = np.concatenate([loud[: 6 * sr], np.zeros(6 * sr, np.float32)])  # half active
print(f"activity ratio (sustained vocal): {vocal_activity_ratio(loud, sr):.2f}")
print(f"activity ratio (half silent):     {vocal_activity_ratio(half, sr):.2f}")

# the screen: only 1 clip clears 0.20 but we need 3 -> the pre-registered fallback to 0.10 fires.
ratios = {"a": 0.5, "b": 0.12, "c": 0.15, "d": 0.05}
res = screen_by_activity(ratios, keep=3, threshold=0.20, fallback=0.10)
print(f"kept={res.kept}  used_threshold={res.used_threshold}  fell_back={res.fell_back}")

## 7 · Pseudo-stem gallery — teacher labels vs real stems (⚠️ RUN LATER)

What do teacher labels *look like*? RUN LATER, this renders spectrogram pairs (teacher pseudo-vocal
vs a real MUSDB vocal) so the teacher's characteristic errors — vocal↔other bleeding, over-
suppression (the THEORY §2 residual $r$) — are visible before training. The layout below is a
**schematic** placeholder (clearly labeled) so the plotting logic is present un-run.

In [ ]:
# ⚠️ RUN THIS LATER — the pseudo-stem gallery (reads the teacher shards written in §4).
# The layout is SCHEMATIC (not real spectrograms) so the figure code is visible un-run. RUN LATER:
# replace the noise panels with STFT log-magnitudes of (teacher pseudo-vocal, real MUSDB vocal).
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(8, 4.5))
demo = np.random.default_rng(1).standard_normal((2, 2, 40, 80))
titles = [["teacher pseudo-vocal (FMA clip)", "real MUSDB vocal (reference)"],
          ["teacher: note vocal↔other bleed", "real: clean studio stem"]]
for r in range(2):
    for c in range(2):
        axes[r][c].imshow(demo[r][c], aspect="auto", origin="lower", cmap="magma")
        axes[r][c].set_title(titles[r][c], fontsize=8); axes[r][c].set_xticks([]); axes[r][c].set_yticks([])
fig.suptitle("SCHEMATIC gallery (RUN LATER → real spectrograms) — teacher error is visible here", fontsize=9)
fig.tight_layout(); plt.show()

## 8 · The leakage-guard story (§3.3) — the pseudo pool structurally refuses MUSDB roots

The MUSDB test set is **never** teacher-labeled and no FMA audio enters any MUSDB split. The
guarantee is **structural**: `PseudoLabeledShards` refuses at construction (a dedicated
`MusdbShardLeak`) if its root is at/under a MUSDB shard root (path containment) or looks like a
MUSDB *decode* root (a 4-stem layout / a `subset` index) — mirroring Direction 06's guard. The
committed **provenance file** closes the audit trail.

In [ ]:
# CPU-runnable now: the MUSDB-path guard raises by construction (no data needed).
import tempfile, json
from singnet.data import PseudoLabeledShards, MusdbShardLeak
from teacher_label import TeacherProvenance, write_provenance

with tempfile.TemporaryDirectory() as d:
    musdb_root = Path(d) / "musdb_shards"; (musdb_root / "pseudo").mkdir(parents=True)
    try:
        PseudoLabeledShards(musdb_root / "pseudo", musdb_roots=(musdb_root,))
        print("LEAK! (guard failed to fire)")
    except MusdbShardLeak as exc:
        print(f"guard fired (nesting under MUSDB refused): {type(exc).__name__}")

    # a clean, separate pseudo root is accepted; the provenance file is the committed audit trail.
    prov = TeacherProvenance(demucs_version="4.0.1", model="htdemucs",
                             settings={"two_stem": "vocals; accompaniment = mixture - vocals"},
                             n_labeled=1200, n_kept=800, used_threshold=0.20,
                             consistency_residual_db_mean=-38.4)
    write_provenance(Path(d) / "provenance.json", prov)
    print("provenance.json:", json.loads((Path(d) / "provenance.json").read_text()))

## 9 · Conclusion — what gate G1 checks

At the teacher session (G1): labeling runs end to end; the **provenance file** is written; the
**consistency residual** is recorded; **≥ 800 clips** survive the activity screen (else the
pre-registered 10 % fallback); and a spot-listen of 3 clips confirms the teacher output is sane.
The manifest (G0b) is frozen and committed — IDs + licenses only, **never audio**. With the data
engine audited, [`02_distillation_experiments.ipynb`](02_distillation_experiments.ipynb) runs the
three data arms and reads the gap-closure ladder.